# DirectQuery Aggregation Advisor

Analyze read-only Fabric monitoring events to identify evidence-based Power BI user-defined aggregation candidates.

- Reads `DirectQueryEnd` events from the selected Fabric monitoring KQL database.
- Does not execute captured SQL or modify semantic models, monitoring data, or source systems.
- Requires Contributor or higher access to the monitored workspace.
- No Lakehouse attachment is required.


## 1. Install the dependency

For scheduled or production runs, install `sqlglot==30.18.0` in the attached Fabric Environment. The next cell provides a session-scoped install for interactive use.


In [ ]:
%pip install sqlglot==30.18.0 --quiet


## 2. Configure the analysis

Set the Fabric monitoring KQL Query URI and database name. Leave `workspace_id = None` to analyze the current notebook workspace, and leave `model_ids` empty to include every observed semantic model.


In [ ]:
# Fabric pipeline parameters
kusto_uri = "https://<monitoring-query-uri>"
monitoring_database = "<monitoring-database-name>"
workspace_id = None
model_ids = []
lookback_days = 30
chunk_hours = 24
dialect = "tsql"
dialect_by_model = {}
model_catalog = []
include_query_text = False


## 3. Load the advisor implementation

In Fabric, add `UDFReview.py` to the notebook's built-in resources. When run from this repository, the loader locates the tested source module automatically.


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

module_candidates = []
try:
    module_candidates.append(Path(notebookutils.nbResPath) / "builtin" / "UDFReview.py")
except (NameError, AttributeError):
    pass

for parent in (Path.cwd(), *Path.cwd().parents):
    module_candidates.append(
        parent / ".github" / "skills" / "power-bi-performance-analyzer" / "references" / "UDFReview.py"
    )

advisor_path = next((path for path in module_candidates if path.is_file()), None)
if advisor_path is None:
    raise FileNotFoundError(
        "UDFReview.py was not found. Add it to the Fabric notebook's built-in resources."
    )

module_spec = spec_from_file_location("UDFReview", advisor_path)
if module_spec is None or module_spec.loader is None:
    raise ImportError(f"Unable to load advisor module from {advisor_path}")

UDFReview = module_from_spec(module_spec)
module_spec.loader.exec_module(UDFReview)
print(f"Loaded advisor: {advisor_path}")


## 4. Run the monitoring analysis

The analysis uses the current user's Fabric credentials and reads the selected monitoring database in chunks. It rejects missing parameters, unsafe endpoints, incomplete reads, and monitoring windows longer than 30 days.


In [ ]:
required_parameters = {
    "kusto_uri": kusto_uri,
    "monitoring_database": monitoring_database,
}
missing_parameters = [
    name for name, value in required_parameters.items()
    if not isinstance(value, str) or not value.strip() or "<" in value
]
if missing_parameters:
    raise ValueError("Set these parameters before running: " + ", ".join(missing_parameters))

report = UDFReview.run_in_fabric(
    spark,
    notebookutils,
    kusto_uri,
    monitoring_database=monitoring_database,
    workspace_id=workspace_id,
    model_ids=model_ids,
    lookback_days=lookback_days,
    chunk_hours=chunk_hours,
    dialect=dialect,
    dialect_by_model=dialect_by_model,
    model_catalog=model_catalog,
    include_query_text=include_query_text,
    progress=print,
)


## 5. Review the summary

Confirm the selected monitoring database, UTC window, observed event count, model count, and candidate totals before interpreting detailed evidence.


In [ ]:
import json

print(json.dumps(report["summary"], indent=2))


## 6. Inspect candidates and supporting evidence

Tables are sorted by associated cumulative DirectQuery duration where available. Coverage values are conservative SQL-shape estimates, not verified aggregation hits.


In [ ]:
import pandas as pd

sections = (
    "model_summary",
    "candidates",
    "table_evidence",
    "column_evidence",
    "query_diagnostics",
)
for section in sections:
    print(f"\n{section}")
    table = pd.DataFrame(report[section])
    if table.empty:
        print("No rows")
        continue
    if "TotalDurationMs" in table.columns:
        table = table.sort_values("TotalDurationMs", ascending=False)
    display(table)

mapping_rows = [
    {
        "CandidateId": candidate["CandidateId"],
        "ItemId": candidate["ItemId"],
        "SourceTable": candidate["SourceTable"],
        "SuggestedAggregationTable": candidate["SuggestedAggregationTable"],
        "ModelTable": candidate["ModelTable"],
        **mapping,
    }
    for candidate in report["candidates"]
    for mapping in candidate["Mappings"]
]
if mapping_rows:
    print("\nCandidate column mappings")
    display(pd.DataFrame(mapping_rows))


## 7. Read the limitations

Treat every candidate as a validation lead. Source-to-model lineage, storage mode, relationships, RLS, source cardinality, refresh cost, and actual aggregation hits still require verification.


In [ ]:
for limitation in report["limitations"]:
    print(f"- {limitation}")
